## 02 – Data Explore

This notebook defines a small suite of utilities for inspecting the raw Night‑2 PSG / Hypnogram EDF files that were fetched in `01_data_fetching` and then applies them to the `physionet-sleep-data` directory.

The goal is to get a quick, reproducible summary of:

* per‑recording metadata (sampling rate, channel list, duration, …)
* hypnogram label distributions
* dataset‑level sleep stage counts per subject
* simple visual checks using an example EEG channel with aligned hypnogram

### Objectives

* Validate file discovery and PSG/hypnogram pairing.
* Inspect a representative PSG file’s channel configuration.
* Analyze a sample hypnogram to confirm label taxonomy and duration.
* Produce dataset‑wide stage counts and save them to a report.
* Provide at least one EEG/hypnogram plot for visual sanity checks.

### Assumptions validated

* All Night‑2 recordings can be found under `SLEEP_EDF_DIR` with predictable naming.
* Hypnogram annotations can be converted to epoch counts (30‑s epochs).
* Channel names and sampling rates are consistent across recordings.
* No obvious missing or corrupted files prevent the pairing logic from running.

### Output

* Printed summaries for individual PSG and hypnogram files.
* Matplotlib figure showing EEG + hypnogram for one subject.
* DataFrame `dataset_stats` with per‑subject epoch totals.
* Plain‑text exploration report written to `reports/02_data_exploration.txt`.

No actual signal preprocessing or relabeling is performed here; this notebook only gathers information to inform subsequent notebooks.

In [ ]:
# --- IMPORTS ---

import json
import warnings
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import mne

from typing_extensions import Literal

In [ ]:
# --- Configuration for data paths and logging ---

# Project-relative data root
DATA_DIR = (Path.cwd() / ".." / "data" ).resolve()
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory {DATA_DIR} does not exist. Please create it before running this notebook.")

# Sleep-EDF directory used by this project (matches downstream notebooks)
SLEEP_EDF_DIR = DATA_DIR / "physionet-sleep-data"
if not SLEEP_EDF_DIR.exists():
    raise FileNotFoundError(f"Sleep-EDF directory {SLEEP_EDF_DIR} does not exist. Please create it before running this notebook.")  

# Report directory for this notebook (matches downstream notebooks)
REPORT_DIR = (Path.cwd() / ".." / "data" / "reports").resolve()
if not REPORT_DIR.exists():
    raise FileNotFoundError(f"Report directory {REPORT_DIR} does not exist. Please create it before running this notebook.")

NOTEBOOK_NAME = "02_data_explore" # This should match the name of this notebook (without the .ipynb extension)
LOG_FILE = REPORT_DIR / f"{NOTEBOOK_NAME}.txt"

# Define a type for the EDF file types 
EdfKind  = Literal["psg", "hypnogram",]

print(f"Data folder: {DATA_DIR}")
print(f"Sleep-EDF folder: {SLEEP_EDF_DIR}")
print(f"Report folder: {REPORT_DIR}")
print(f"Log file: {LOG_FILE}")

In [ ]:
def get_sample_edf_file(path: str, kind: EdfKind) -> Path:
    p = Path(path).resolve()

    # 1) Validate path existence
    if not p.exists():
        raise FileNotFoundError(f"Path does not exist: {p}")

    # 2) Define matcher for file name
    def matches_kind(f: Path) -> bool:
        name = f.name.lower()
        if kind == "hypnogram":
            return "hypnogram" in name
        if kind == "psg":
            return "hypnogram" not in name
        # (Should be unreachable due to Literal typing, but kept defensive)
        raise ValueError(f"Unsupported kind: {kind!r} (expected 'psg' or 'hypnogram')")

    # 3) If directory: search for EDFs, then filter by kind
    if p.is_dir():
        edf_files = sorted(
            f for f in p.iterdir()
            if f.is_file() and f.suffix.lower() == ".edf"
        )

        if not edf_files:
            raise FileNotFoundError(f"No .edf files found in directory: {p}")

        candidates = sorted(f for f in edf_files if matches_kind(f))

        if not candidates:
            raise FileNotFoundError(
                f"No {kind} EDF files found in directory: {p}"
            )

        return candidates[0]

    # 4) If file: validate it directly
    if p.is_file():
        if p.suffix.lower() != ".edf":
            raise ValueError(f"File is not an EDF file: {p}")

        if not matches_kind(p):
            raise ValueError(f"Provided EDF file does not match kind={kind!r}: {p}")

        return p

    # 5) Anything else (rare): invalid path type
    raise ValueError(f"Invalid path provided (not a file or directory): {p}")

In [ ]:
def get_edf_file_pairs(data_root: Path) -> list[list[str]]:
    """
    Search `data_root` for PSG files and their matching hypnograms.

    Parameters
    ----------
    data_root : pathlib.Path
        Directory holding the raw Sleep‑EDF files.

    Returns
    -------
    list[list[str]]
        A list of two‑element lists; each inner list contains the
        stringified path to a PSG file and the path to the corresponding
        Hypnogram file.
    """
    pairs: list[list[str]] = []  # accumulate [[psg_path, hyp_path], …]

    for psg in sorted(data_root.glob("*2E0-PSG.edf")):
        subject_id = psg.name[:5]

        hyp = list(data_root.glob(f"{subject_id}*Hypnogram.edf"))
        if hyp:
            pairs.append([str(psg), str(hyp[0])])

    return pairs



In [ ]:
def write_exploration_report(report: pd.DataFrame, out_path: str | Path) -> str:
    """
    Write a subject‑level summary to disk as a formatted plain‑text table.

    The output is a tab‑separated plain‑text file where the first line contains 
    column headers, the second line is a row of dashes, and subsequent lines are 
    the data. Each column is given a fixed width computed from the longest entry 
    in that column plus one character of slack, rounded up to the next multiple 
    of four for alignment.

    Parameters
    ----------
    report : pd.DataFrame
        One row per subject with columns like 'Patient_ID', 'Movement time', 
        'Sleep stage 1', …, 'Sleep stage W', etc. Should be pre‑aggregated 
        (e.g., epoch counts per subject per stage).
    out_path : str or pathlib.Path
        Path to write the output file to. A string is automatically converted 
        to pathlib.Path internally.

    Returns
    -------
    str
        Absolute path of the file that was written.
    """
    # Convert to Path object and ensure parent directories exist
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Convert all data to strings to enable length measurement
    str_df = report.astype(str)

    # Compute column widths: max of header length and data length, plus slack, rounded to multiple of 4
    widths = {}
    for col in str_df.columns:
        maxlen = max(str_df[col].map(len).max(), len(col))
        w = maxlen + 1                    # Add one character of slack
        widths[col] = ((w + 3) // 4) * 4  # Round up to next multiple of 4

    # Build header and separator lines with tab‑separated columns
    header = "\t".join(col.ljust(widths[col]) for col in str_df.columns)
    sep    = "\t".join("-" * widths[col]        for col in str_df.columns)

    # Accumulate all lines: header, separator, then data rows
    lines = [header, sep]
    for _, row in str_df.iterrows():
        lines.append("\t".join(row[col].ljust(widths[col]) for col in str_df.columns))

    # Write to file and return resolved absolute path
    out_path.write_text("\n".join(lines))
    return str(out_path.resolve())


In [ ]:
def explore_edf_file(edf_path):
    """
    If edf_path is a directory, pick the first PSG .edf file inside (sorted
    order).  Otherwise treat edf_path as a file.  Return basic metadata for PSG
    or Hypnogram.  Formats output as requested and returns JSON packet.

    The caller expects to inspect PSG recordings; if a directory contains no
    non‑hypnogram EDF files an error is raised and execution stops.
    """
    warnings.filterwarnings("ignore", category=RuntimeWarning)

    target = get_sample_edf_file(edf_path, kind="psg").resolve()

    file_name = target.name
    # file_type = "Hypnogram" if "Hypnogram" in file_name else "PSG"

    raw = mne.io.read_raw_edf(str(target), preload=False, verbose=False)
    sfreq = float(raw.info.get("sfreq", np.nan))
    duration_s = (
        float(raw.n_times) / sfreq
        if raw.n_times and sfreq and sfreq > 0
        else 0.0
    )

    # Build channel table
    channels_data = []
    for idx, ch_name in enumerate(raw.ch_names):
        channels_data.append(
            {
                "Channel_Number": idx + 1,
                "Channel_Name": ch_name,
                "Sampling_Frequency": sfreq,
                "Filter_Cutoff": "N/A",
                "Length_Seconds": duration_s,
            }
        )

    # Print formatted output
    print("---- EDF DATA STATISTICS ----")
    print(f"Folder Location: {target.parent}")
    print(f"Sample File Name: {file_name}")
    print(
        f"{'Channel #':<12} {'Channel Name':<20} "
        f"{'Sampling Freq':<16} {'Filter Cutoff':<14} "
        f"{'Length (s)':<12}"
    )
    print("-" * 74)
    for ch in channels_data:
        print(
            f"{ch['Channel_Number']:<12} {ch['Channel_Name']:<20} "
            f"{ch['Sampling_Frequency']:<16.1f} {ch['Filter_Cutoff']:<14} "
            f"{ch['Length_Seconds']:<12.1f}"
        )
    print("-" * 74)

    # Return JSON packet
    result = {
        "folder_location": str(target.parent),
        "sample_file_name": file_name,
        "num_channels": len(raw.ch_names),
        "channels": channels_data,
    }
    return result

In [ ]:
def explore_hypnogram_file(hypnogram_path):
    """
    Analyze sleep stage distribution from a hypnogram EDF file or from a
    directory containing one (or more) such files.

    Parameters
    ----------
    hypnogram_path : str or pathlib.Path
        Path to a hypnogram EDF file **or** to a directory.  When a
        directory is supplied the first matching ``*Hypnogram.edf`` file
        (sorted lexicographically) is chosen.  If no file can be found a
        ``FileNotFoundError`` is raised.

    Returns
    -------
    pd.DataFrame
        DataFrame with sleep stage statistics (stage name and epoch count).
    """
    target_file = get_sample_edf_file(hypnogram_path, kind="hypnogram").resolve()

    # read and convert annotations
    annot = mne.read_annotations(str(target_file))
    df_annot = annot.to_data_frame()
    df_annot["epochs"] = df_annot["duration"] / 30
    stats = df_annot.groupby("description")["epochs"].sum().reset_index()
    stats.columns = ["SLEEP STAGE", "EPOCH COUNT"]

    print("---- HYPNOGRAM DATA STATISTICS ----")
    print(f"Folder Location: {target_file.parent}")
    print(f"Sample File Name: {target_file.name}")
    print()
    print(stats.to_string(index=False, justify="left", col_space=16))

    return stats

In [ ]:
def plot_eeg_with_hypnogram(
    psg_path,
    hypno_path,
    stage_annots=None,
    eeg_pick=None,
    epoch_len_s=30,
    t0_s=0,
    t1_s=None,
    hypno_unknown_label="UNKNOWN",
):
    """
    Plot one EEG channel and an aligned hypnogram.

    psg_path / hypno_path : pathlib.Path/str
        Path (or filename relative to SLEEP_EDF_DIR) to the PSG and hypnogram EDF files.
    """
    psg_path = Path(psg_path)
    hypno_path = Path(hypno_path)

    # try resolving relative to SLEEP_EDF_DIR if files not found as given
    if not psg_path.exists() and SLEEP_EDF_DIR is not None:
        candidate = Path(SLEEP_EDF_DIR) / psg_path
        if candidate.exists():
            psg_path = candidate
    if not hypno_path.exists() and SLEEP_EDF_DIR is not None:
        candidate = Path(SLEEP_EDF_DIR) / hypno_path
        if candidate.exists():
            hypno_path = candidate

    if not psg_path.exists() or not hypno_path.exists():
        raise FileNotFoundError(f"Could not find PSG/Hypnogram files: {psg_path}, {hypno_path}")

    raw = mne.io.read_raw_edf(str(psg_path), preload=True, verbose=False)
    annot = mne.read_annotations(str(hypno_path))
    raw.set_annotations(annot, emit_warning=False)

    # ---------- Resolve EEG channel ----------
    ch_names = list(raw.ch_names)
    if eeg_pick is None:
        eeg_candidates = [ch for ch in ch_names if "EEG" in ch.upper()]
        eeg_pick = eeg_candidates[0] if len(eeg_candidates) > 0 else ch_names[0]

    if eeg_pick not in ch_names:
        raise ValueError(f"eeg_pick='{eeg_pick}' not found in raw.ch_names.")

    # ---------- Resolve time window ----------
    sfreq = float(raw.info["sfreq"])
    rec_len_s = float(raw.times[-1]) if raw.n_times > 1 else 0.0
    if t1_s is None:
        t1_s = rec_len_s
    t0_s = max(0.0, float(t0_s))
    t1_s = min(float(t1_s), rec_len_s)
    if t1_s <= t0_s:
        raise ValueError("t1_s must be > t0_s within the recording duration.")

    # ---------- Pull EEG data ----------
    start_samp = int(round(t0_s * sfreq))
    stop_samp = int(round(t1_s * sfreq))
    data, times = raw.get_data(picks=[eeg_pick], start=start_samp, stop=stop_samp, return_times=True)
    eeg = data[0]
    times = times  # seconds (relative)

    # ---------- Build stage dataframe if not provided ----------
    if stage_annots is None:
        if raw.annotations is None or len(raw.annotations) == 0:
            stage_annots = pd.DataFrame(columns=["onset", "duration", "stage"])
        else:
            a = raw.annotations
            stage_annots = pd.DataFrame(
                {
                    "onset": np.array(a.onset, dtype=float),
                    "duration": np.array(a.duration, dtype=float),
                    "stage": np.array(a.description, dtype=str),
                }
            )

    for col in ["onset", "duration", "stage"]:
        if col not in stage_annots.columns:
            raise ValueError(f"stage_annots must contain column '{col}'.")

    df_stage = stage_annots.copy()

    # ---------- Normalize stage onsets to seconds-from-raw-start ----------
    onset_dtype = df_stage["onset"].dtype

    if np.issubdtype(onset_dtype, np.number):
        onset_s = df_stage["onset"].astype(float)
        end_s = onset_s + df_stage["duration"].astype(float)
    else:
        onset_dt = pd.to_datetime(df_stage["onset"], errors="coerce")
        if onset_dt.isna().all():
            raise ValueError("stage_annots['onset'] could not be parsed as datetime-like.")
        ref_dt = onset_dt.iloc[0]
        onset_s = (onset_dt - ref_dt).dt.total_seconds()
        end_s = onset_s + df_stage["duration"].astype(float)

    df_stage["_onset_s"] = onset_s.to_numpy()
    df_stage["_end_s"] = end_s.to_numpy()
    df_stage["_stage"] = df_stage["stage"].astype(str).to_numpy()

    # ---------- Create epoch midpoints for hypnogram ----------
    epoch_edges = np.arange(0, rec_len_s + epoch_len_s, epoch_len_s, dtype=float)
    epoch_mid = epoch_edges[:-1] + (epoch_len_s / 2.0)

    starts = df_stage["_onset_s"]
    ends = df_stage["_end_s"]
    stages = df_stage["_stage"]

    order = np.argsort(starts)
    starts = np.asarray(starts)[order]
    ends = np.asarray(ends)[order]
    stages = np.asarray(stages)[order]

    hypno = np.array([hypno_unknown_label] * len(epoch_mid), dtype=object)
    if len(starts) > 0:
        idx = np.searchsorted(ends, epoch_mid, side="right")
        valid = (idx < len(starts)) & (epoch_mid >= starts[idx])
        hypno[valid] = stages[idx[valid]]

    # ---------- Restrict hypnogram to plotting window ----------
    win_mask = (epoch_mid >= t0_s) & (epoch_mid <= t1_s)
    hypno_t = epoch_mid[win_mask]
    hypno_lbl = hypno[win_mask]

    # ---------- Map stage labels to numeric levels for plotting ----------
    uniq = [x for x in pd.unique(hypno_lbl) if pd.notna(x)]
    if hypno_unknown_label not in uniq:
        uniq = [hypno_unknown_label] + uniq

    stage_to_y = {lab: i for i, lab in enumerate(uniq)}
    hypno_y = np.array([stage_to_y.get(lab, stage_to_y[hypno_unknown_label]) for lab in hypno_lbl], dtype=float)

    # ---------- Plot ----------
    fig, (ax_eeg, ax_hyp) = plt.subplots(
        2, 1, figsize=(14, 6), sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05}
    )

    ax_eeg.plot(times, eeg, linewidth=0.6)
    ax_eeg.set_ylabel(f"{eeg_pick}\n(µV or raw units)")
    ax_eeg.set_title("EEG channel + Hypnogram")

    ax_hyp.step(hypno_t, hypno_y, where="mid")
    ax_hyp.set_yticks(list(stage_to_y.values()))
    ax_hyp.set_yticklabels(list(stage_to_y.keys()))
    ax_hyp.set_xlabel("Time (s from start)")
    ax_hyp.set_ylabel("Stage")

    ax_hyp.set_xlim(t0_s, t1_s)

    plt.show()
    return fig, (ax_eeg, ax_hyp)

In [ ]:
def build_dataset_stats(raw_dir):

    pairs = get_edf_file_pairs(raw_dir)

    subject_data_list = []

    print("--- Building Dataset Statistics ---")
    
    for psg_path, hypno_path in pairs:
        filename = Path(psg_path).name
        subject_id = filename[:5]

        try:
            # 1. Extract the Subject ID (e.g., SC402)
            subject_id = filename[:5]

            # 2. Load Annotations
            annot = mne.read_annotations(hypno_path)

            # 4. Calculate Epochs
            df_temp = annot.to_data_frame()
            df_temp["epochs"] = df_temp["duration"] / 30.0

            # 5. Summarize
            subject_summary = df_temp.groupby("description")["epochs"].sum().to_dict()
            subject_summary["Patient_ID"] = subject_id

            subject_data_list.append(subject_summary)
            print("#", end="")

        except Exception as e:
            raise RuntimeError(f"Error processing subject {filename}: {e}") from e

    # Create the Master Table
    subject_table = pd.DataFrame(subject_data_list)
    if subject_table.empty:
        return subject_table

    cols = ["Patient_ID"] + [c for c in subject_table.columns if c != "Patient_ID"]
    subject_table = subject_table[cols].fillna(0)

    return subject_table

In [ ]:
# --- EXPLORE PSG FILE ---
result = explore_edf_file(SLEEP_EDF_DIR)
#print("\n--- JSON PACKET ---")
#print(json.dumps(result, indent=4))

In [ ]:
# --- EXPLORE HYPNOGRAM FILE ---
stats = explore_hypnogram_file(SLEEP_EDF_DIR)

In [ ]:
# --- PLOT SAMPLE EEG WITH HYPNOGRAM ---
psg_file, hypno_file = get_edf_file_pairs(SLEEP_EDF_DIR)[0]  # Get the first pair for demonstration
fig, (ax_eeg, ax_hyp) = plot_eeg_with_hypnogram(psg_file, hypno_file)

In [ ]:
# --- BUILD DATASET STATISTICS ---
dataset_stats = build_dataset_stats(SLEEP_EDF_DIR)

In [ ]:
# --- SAVE EXPLORATION REPORT ---
report_path = write_exploration_report(dataset_stats, LOG_FILE)
print(f"Exploration report saved to: {report_path}")